# Create & publish a Fabric data agent (preview)

Creates a Fabric data agent over the demo lakehouse and **publishes** it so a
Microsoft Foundry agent can consume it as a knowledge source.

> Uses the preview `fabric-data-agent-sdk`, which runs **only inside a Fabric
> notebook**. Method signatures may change while the SDK is in preview.

Parameters (substituted at deploy time): `DATA_AGENT_NAME`, `LAKEHOUSE_NAME`, `WORKSPACE_ID`.

In [ ]:
%pip install -q fabric-data-agent-sdk

In [ ]:
# Parameters (replaced by the deployer; defaults keep the notebook runnable standalone)
DATA_AGENT_NAME = "{{DATA_AGENT_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
WORKSPACE_ID = "{{WORKSPACE_ID}}"

if DATA_AGENT_NAME.startswith("{{"):
    DATA_AGENT_NAME = "analytics_data_agent"
if LAKEHOUSE_NAME.startswith("{{"):
    LAKEHOUSE_NAME = "analytics_lakehouse"

print(f"Data agent: {DATA_AGENT_NAME}")
print(f"Lakehouse : {LAKEHOUSE_NAME}")

In [ ]:
# Create the data agent, add the lakehouse as a data source, and publish.
# The SDK is in preview; we guard each step so a single API change doesn't
# leave the agent half-configured without a clear message.
from fabric.dataagent.client import create_data_agent, FabricDataAgentManagement

data_agent = create_data_agent(DATA_AGENT_NAME)
mgmt = FabricDataAgentManagement(data_agent)

# Ground the agent on the demo lakehouse (NL2SQL over its gold tables).
try:
    mgmt.add_datasource(LAKEHOUSE_NAME, type="lakehouse")
except TypeError:
    # Older/newer SDK signature fallback.
    mgmt.add_datasource(LAKEHOUSE_NAME)

# A short instruction improves NL2SQL accuracy.
try:
    mgmt.update_configuration(
        instructions=(
            "You answer questions about manufacturing quality-control data "
            "(production batches, sensor readings, equipment, defects). "
            "Prefer the gold tables in the lakehouse."
        )
    )
except Exception as e:  # noqa: BLE001 - configuration is best-effort
    print(f"(configuration step skipped: {e})")

In [ ]:
# Publish — required before Foundry can consume the agent.
mgmt.publish()

published_url = f"https://fabric.microsoft.com/groups/{WORKSPACE_ID}/aiskills/{DATA_AGENT_NAME}"
print("Published Fabric data agent.")
print(f"Endpoint (by name): {published_url}")

# Hand the agent name back to the deployer, which resolves the artifact id by
# listing workspace items of type 'DataAgent'.
try:
    import notebookutils  # noqa: F401
    notebookutils.notebook.exit(DATA_AGENT_NAME)
except Exception:  # noqa: BLE001 - exit is best-effort
    pass